In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 1. Veri Anlama

Bu bölüm, herhangi bir ön işleme veya makine öğrenmesi tekniği uygulamadan önce veri setinin yapısını ve özelliklerini anlamaya odaklanır.

In [23]:
data_path = Path("../data/raw")
list(data_path.iterdir())

[PosixPath('../data/raw/E Commerce Dataset.xlsx')]

In [24]:
file_path = data_path / "E Commerce Dataset.xlsx"
df = pd.read_excel(file_path, sheet_name="E Comm")
df.head()
df.shape

(5630, 20)

### 2. Veri Setine Genel Bakış

Bu bölüm; değişkenleri, veri tipleri ve genel özellikleri dahil olmak üzere veri setinin yapısını inceler.

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   str    
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   str    
 7   Gender                       5630 non-null   str    
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   str    
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   str    
 13  NumberOfAddress              

In [26]:
missing_values = pd.DataFrame({
	"missing_count" : df.isnull().sum(),
	"missing_percentage" : (df.isnull().sum() / len(df)) * 100
})
missing_values = missing_values[missing_values["missing_count"] > 0]

missing_values.sort_values(
	by= "missing_percentage",
	ascending=False
)

,missing_count,missing_percentage
DaySinceLastOrder,307,5.452931
OrderAmountHikeFromlastYear,265,4.706927
Tenure,264,4.689165
OrderCount,258,4.582593
CouponUsed,256,4.547069
HourSpendOnApp,255,4.529307
WarehouseToHome,251,4.458259


#### Eksik Değerlerin Veri Seti Üzerindeki Etkisi

Bu analiz, eksik gözlemlerin müşteri kayıtları arasında nasıl dağıldığını ve eksik veriye sahip müşterilerin tam kayıtlara sahip müşterilerden farklı bir terk (churn) oranı sergileyip sergilemediğini inceler.

In [27]:
row_missing = df.isnull().any(axis=1)

missing_impact = pd.DataFrame({
    "Customer Group": ["With Missing Values", "Without Missing Values"],
    "Customer Count": [
        row_missing.sum(),
        (~row_missing).sum()
    ],
    "Percentage": [
        row_missing.mean() * 100,
        (~row_missing).mean() * 100
    ],
    "Churn Rate": [
        df.loc[row_missing, "Churn"].mean() * 100,
        df.loc[~row_missing, "Churn"].mean() * 100
    ]
})

missing_impact

,Customer Group,Customer Count,Percentage,Churn Rate
0,With Missing Values,1856,32.966252,17.079741
1,Without Missing Values,3774,67.033748,16.719661


In [28]:
missing_per_row = df.isnull().sum(axis=1)

missing_per_row.value_counts().sort_index()

0    3774
1    1856
Name: count, dtype: int64

Sonuçlar, eksik veriye sahip her kaydın yalnızca tek bir eksik değer içerdiğini göstermektedir. Bu kayıtların çıkarılması veri setinin yaklaşık %33'ünün kaybına yol açacağından, bir doldurma (imputation) stratejisine karar vermeden önce kayıtları veri setinde tutacak ve eksikliğin terk (churn) ile ilişkisini inceleyeceğiz.

In [29]:
from scipy.stats import chi2_contingency

results = []

for column in missing_values.index:
    missing = df[column].isnull()

    contingency_table = pd.crosstab(missing, df["Churn"])
    chi2, p_value, _, _ = chi2_contingency(contingency_table)

    results.append({
        "Variable": column,
        "Missing Churn Rate (%)": df.loc[missing, "Churn"].mean() * 100,
        "Non-Missing Churn Rate (%)": df.loc[~missing, "Churn"].mean() * 100,
        "P-value": p_value
    })

churn_missing_analysis = pd.DataFrame(results)
churn_missing_analysis

,Variable,Missing Churn Rate (%),Non-Missing Churn Rate (%),P-value
0,Tenure,30.681818,16.157287,1.257852e-09
1,WarehouseToHome,33.466135,16.062465,1.112029e-12
2,HourSpendOnApp,22.745098,16.558140,1.262861e-02
3,OrderAmountHikeFromlastYear,5.283019,17.409133,4.075383e-07
4,CouponUsed,3.125000,17.491626,3.298690e-09
5,OrderCount,6.976744,17.311988,2.154167e-05
6,DaySinceLastOrder,17.589577,16.795040,7.769376e-01


Sonuçlar, yedi değişkenin beşi için eksiklik ile terk (churn) arasında anlamlı bir ilişki olduğunu göstermektedir. Bu durum, eksik değerlerin tamamen rastgele olmayabileceğini ve ön işleme aşamasında dikkatle ele alınması gerektiğini düşündürmektedir.

#### Kategorik Değişken İncelemesi

Bu bölüm, tutarsız veya beklenmeyen değerleri tespit etmek için kategorik değişkenleri inceler.

In [30]:
categorical_cols = df.select_dtypes(include="object").columns

for column in categorical_cols:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False))


PreferredLoginDevice:
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode:
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

Gender:
Gender
Male      3384
Female    2246
Name: count, dtype: int64

PreferedOrderCat:
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64

MaritalStatus:
MaritalStatus
Married     2986
Single      1796
Divorced     848
Name: count, dtype: int64


/var/folders/z2/d5njtmhx3vgdxy1c5hmh43hc0000gn/T/ipykernel_5617/3337513672.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns


In [31]:
category_mapping = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "CC": "Credit Card",
        "COD": "Cash on Delivery"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

for column, mapping in category_mapping.items():
    df[column] = df[column].replace(mapping)

### Tekrarlanan Kayıtların İncelenmesi

Bu analiz, veri setinde tamamen aynı kayıtların veya tekrarlanan müşteri ID'lerinin bulunup bulunmadığını kontrol eder.

In [32]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerIDs:", df["CustomerID"].duplicated().sum())

Duplicate rows: 0
Duplicate CustomerIDs: 0


### Sayısal Değişkenlerde Aykırı Değerlerin İncelenmesi

Bu analiz, sayısal değişkenlerde aykırı değerlerin bulunup bulunmadığını incelemeyi amaçlar.

In [33]:
numeric_cols = df.select_dtypes(include="number").columns.drop("CustomerID")

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Churn,5630.0,0.168384,0.374240,0.0,0.00,0.00,0.0000,1.00
Tenure,5366.0,10.189899,8.557241,0.0,2.00,9.00,16.0000,61.00
CityTier,5630.0,1.654707,0.915389,1.0,1.00,1.00,3.0000,3.00
WarehouseToHome,5379.0,15.639896,8.531475,5.0,9.00,14.00,20.0000,127.00
HourSpendOnApp,5375.0,2.931535,0.721926,0.0,2.00,3.00,3.0000,5.00
NumberOfDeviceRegistered,5630.0,3.688988,1.023999,1.0,3.00,4.00,4.0000,6.00
SatisfactionScore,5630.0,3.066785,1.380194,1.0,2.00,3.00,4.0000,5.00
NumberOfAddress,5630.0,4.214032,2.583586,1.0,2.00,3.00,6.0000,22.00
Complain,5630.0,0.284902,0.451408,0.0,0.00,0.00,1.0000,1.00
OrderAmountHikeFromlastYear,5365.0,15.707922,3.675485,11.0,13.00,15.00,18.0000,26.00


### Aykırı Değerlerin Tespiti

Sayısal değişkenlerde IQR yöntemi kullanılarak aykırı değer adayları belirlenir. Bu değerler doğrudan silinmeden önce incelenir.

In [34]:
outlier_cols = [
    "WarehouseToHome",
    "NumberOfAddress",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder"
]

for column in outlier_cols:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[column][
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print(f"{column}: {len(outliers)} outlier candidates")

WarehouseToHome: 2 outlier candidates
NumberOfAddress: 4 outlier candidates
CouponUsed: 629 outlier candidates
OrderCount: 703 outlier candidates
DaySinceLastOrder: 62 outlier candidates


IQR yöntemiyle bazı sayısal değişkenlerde aykırı değer adayları tespit edilmiştir. Ancak bu değerlerin müşteri davranışını temsil etme olasılığı bulunduğundan, doğrulanmış bir veri hatası olmadığı için herhangi bir kayıt silinmemiştir.